# 📚 Technique 57: Hybrid Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/57_hybrid_retrieval.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 57
**Difficulty:** Advanced

## 📋 Description

**Hybrid Retrieval** combines multiple search methods—typically sparse (keyword/BM25) and dense (semantic/vector) retrieval—to leverage the strengths of each approach. This fusion provides more robust and comprehensive results than either method alone, capturing both exact keyword matches and conceptual semantic similarity.

### When to Use:
- When you need both **exact matches** and **conceptual understanding**
- For **diverse query types** (some keyword-heavy, some semantic)
- When dealing with **technical terminology** and **natural language**
- For **production RAG systems** requiring high recall
- When you want to **reduce false negatives** from either method alone

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                   HYBRID RETRIEVAL PIPELINE                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│                         USER QUERY                              │
│                              │                                  │
│              ┌───────────────┴───────────────┐                  │
│              ▼                               ▼                  │
│  ┌──────────────────────┐    ┌──────────────────────┐          │
│  │   SPARSE RETRIEVAL   │    │   DENSE RETRIEVAL    │          │
│  │   (Keyword/BM25)     │    │   (Semantic/Vector)  │          │
│  ├──────────────────────┤    ├──────────────────────┤          │
│  │ • Exact term matches │    │ • Concept similarity │          │
│  │ • ID/code lookup     │    │ • Synonym handling   │          │
│  │ • Fast inverted idx  │    │ • Context understanding│        │
│  └──────────┬───────────┘    └──────────┬───────────┘          │
│             │                           │                       │
│             ▼                           ▼                       │
│  ┌──────────────────────┐    ┌──────────────────────┐          │
│  │  Results A:          │    │  Results B:          │          │
│  │  [Doc1: 0.9]         │    │  [Doc3: 0.85]        │          │
│  │  [Doc2: 0.7]         │    │  [Doc1: 0.8]         │          │
│  │  [Doc4: 0.5]         │    │  [Doc5: 0.75]        │          │
│  └──────────┬───────────┘    └──────────┬───────────┘          │
│             │                           │                       │
│             └───────────┬───────────────┘                       │
│                         ▼                                       │
│              ┌──────────────────────┐                          │
│              │    FUSION STEP       │                          │
│              ├──────────────────────┤                          │
│              │ Methods:             │                          │
│              │ • Linear Combination │                          │
│              │ • RRF (Reciprocal)   │                          │
│              │ • Weighted Sum       │                          │
│              └──────────┬───────────┘                          │
│                         ▼                                       │
│              ┌──────────────────────┐                          │
│              │  FINAL RANKED RESULTS│                          │
│              │  [Doc1: 0.92]        │                          │
│              │  [Doc3: 0.78]        │                          │
│              │  [Doc2: 0.65]        │                          │
│              └──────────────────────┘                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Fusion Methods:

**1. Linear Combination (Weighted Sum):**
```
score_final = α × score_sparse + (1-α) × score_dense
```

**2. Reciprocal Rank Fusion (RRF):**
```
score_rrf = Σ 1/(k + rank_i)  for each result list
k = constant (typically 60)
```

**3. Convex Combination:**
```
Normalize scores to [0,1], then combine with weights
```

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai numpy scikit-learn rank-bm25

In [ ]:
import os
from getpass import getpass
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import re

# Setup API
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector for text"""
    response = client.embeddings.create(model=model, input=text)
    return np.array(response.data[0].embedding)

def tokenize(text):
    """Simple tokenization for BM25"""
    return re.findall(r'\w+', text.lower())

## 💡 Basic Example

Implementing hybrid retrieval with BM25 + semantic search.

In [ ]:
# Sample document collection
documents = [
    "Python is a high-level programming language created by Guido van Rossum",
    "JavaScript runs in web browsers and enables interactive websites",
    "Python snakes are non-venomous constrictors found in tropical regions",
    "Java is a statically typed language used for enterprise applications",
    "The python programming language is popular for data science and AI",
    "Java coffee originates from the Indonesian island of Java",
    "Python 3.10 introduced pattern matching and better error messages",
    "JavaScript frameworks include React, Angular, and Vue.js"
]

print("Building hybrid search index...\n")

# 1. Build BM25 (sparse) index
tokenized_docs = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
print(f"✓ BM25 index built ({len(documents)} documents)")

# 2. Build semantic (dense) index
doc_embeddings = [get_embedding(doc) for doc in documents]
doc_embeddings = np.array(doc_embeddings)
print(f"✓ Semantic index built ({doc_embeddings.shape[1]} dimensions)\n")

def hybrid_search(query, bm25, doc_embeddings, documents, alpha=0.5, top_k=5):
    """
    Hybrid search combining BM25 and semantic search
    
    Args:
        query: Search query
        bm25: BM25 index
        doc_embeddings: Document embedding matrix
        documents: List of document texts
        alpha: Weight for sparse (BM25) vs dense (semantic) [0-1]
        top_k: Number of results to return
    """
    # BM25 scores
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Normalize BM25 scores to [0, 1]
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()
    
    # Semantic scores
    query_embedding = get_embedding(query)
    semantic_scores = cosine_similarity([query_embedding], doc_embeddings)[0]
    
    # Combine scores
    combined_scores = alpha * bm25_scores + (1 - alpha) * semantic_scores
    
    # Get top-k results
    top_indices = np.argsort(combined_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'document': documents[idx],
            'combined_score': float(combined_scores[idx]),
            'bm25_score': float(bm25_scores[idx]),
            'semantic_score': float(semantic_scores[idx]),
            'index': idx
        })
    
    return results

# Test queries
test_queries = [
    ("programming language", 0.5),  # Balanced
    ("python code", 0.3),           # Favor semantic
    ("Guido van Rossum", 0.7),      # Favor keyword
]

for query, alpha in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}' | Alpha (BM25 weight): {alpha}")
    print(f"{'='*70}")
    
    results = hybrid_search(query, bm25, doc_embeddings, documents, alpha=alpha, top_k=3)
    
    print(f"\n{'Rank':<6}{'Combined':<10}{'BM25':<10}{'Semantic':<10} Document{'':<30}")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        doc_short = r['document'][:40] + "..." if len(r['document']) > 40 else r['document']
        print(f"{i:<6}{r['combined_score']:<10.3f}{r['bm25_score']:<10.3f}{r['semantic_score']:<10.3f} {doc_short}")

## 🌍 Real-World Example

E-commerce product search with hybrid retrieval.

In [ ]:
# E-commerce product catalog
products = [
    {"id": "P001", "name": "iPhone 15 Pro", "description": "Apple smartphone with A17 chip, 256GB storage, titanium design"},
    {"id": "P002", "name": "Samsung Galaxy S24", "description": "Android flagship phone with AI features, 512GB, 200MP camera"},
    {"id": "P003", "name": "MacBook Pro 16", "description": "Apple laptop with M3 Max chip, 32GB RAM, professional creative workstation"},
    {"id": "P004", "name": "Dell XPS 15", "description": "Windows laptop with Intel i9, OLED display, business productivity"},
    {"id": "P005", "name": "Sony WH-1000XM5", "description": "Premium wireless noise-canceling headphones, 30-hour battery"},
    {"id": "P006", "name": "AirPods Pro 2", "description": "Apple wireless earbuds with spatial audio and active noise cancellation"},
    {"id": "P007", "name": "iPad Pro 12.9", "description": "Apple tablet with M2 chip, Liquid Retina XDR display, professional drawing"},
    {"id": "P008", "name": "Apple Watch Ultra 2", "description": "Rugged smartwatch with GPS, health monitoring, 36-hour battery"},
    {"id": "P009", "name": "Nintendo Switch OLED", "description": "Gaming console with 7-inch OLED screen, handheld and docked modes"},
    {"id": "P010", "name": "PlayStation 5", "description": "Sony gaming console with 4K graphics, ray tracing, ultra-fast SSD"}
]

# Prepare product texts
product_texts = [f"{p['name']}: {p['description']}" for p in products]

# Build indexes
print("Building e-commerce search index...\n")
tokenized_products = [tokenize(text) for text in product_texts]
product_bm25 = BM25Okapi(tokenized_products)
product_embeddings = np.array([get_embedding(text) for text in product_texts])
print(f"✓ Indexed {len(products)} products\n")

def search_products(query, alpha=0.5, top_k=5):
    """Search products with hybrid retrieval"""
    results = hybrid_search(query, product_bm25, product_embeddings, product_texts, alpha, top_k)
    
    # Enrich with product details
    for r in results:
        r['product'] = products[r['index']]
    
    return results

# Test different query types
print("=== E-COMMERCE SEARCH TESTS ===\n")

search_tests = [
    ("Apple phone", 0.5, "Brand + category query"),
    ("wireless headphones with long battery", 0.3, "Feature description"),
    ("P001", 0.8, "Product ID lookup"),
    ("gaming console 4K", 0.4, "Mixed features"),
    ("professional creative work", 0.3, "Use case query")
]

for query, alpha, description in search_tests:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"Type: {description} | Alpha: {alpha}")
    print(f"{'='*70}")
    
    results = search_products(query, alpha=alpha, top_k=3)
    
    for i, r in enumerate(results, 1):
        product = r['product']
        print(f"\n{i}. {product['name']} ({product['id']})")
        print(f"   Score: {r['combined_score']:.3f} (BM25: {r['bm25_score']:.3f}, Semantic: {r['semantic_score']:.3f})")
        print(f"   {product['description'][:60]}...")

## ❌ Failure Case

When hybrid retrieval doesn't help and common pitfalls.

In [ ]:
# Demonstrating hybrid retrieval limitations

print("=== FAILURE 1: POOR ALPHA TUNING ===\n")
query = "python programming"

for alpha in [0.0, 0.5, 1.0]:
    results = hybrid_search(query, bm25, doc_embeddings, documents, alpha=alpha, top_k=3)
    print(f"Alpha = {alpha} ({'Semantic only' if alpha==0 else 'BM25 only' if alpha==1 else 'Balanced'})")
    for r in results[:2]:
        print(f"  {r['combined_score']:.3f}: {r['document'][:50]}...")
    print()

print("⚠️ Issue: Extreme alpha values lose benefits of hybrid approach\n")

print("=== FAILURE 2: SCORE NORMALIZATION PROBLEMS ===\n")

# Documents with very different BM25 score ranges
short_docs = ["cat", "dog", "cat dog"]
long_docs = ["The domestic cat is a small carnivorous mammal", "Dogs are domesticated mammals", "Cats and dogs are popular pets"]

tokenized_short = [tokenize(d) for d in short_docs]
bm25_short = BM25Okapi(tokenized_short)

query_short = "cat"
bm25_scores_short = bm25_short.get_scores(tokenize(query_short))
print(f"Short docs BM25 scores: {bm25_scores_short}")
print(f"Score range: {bm25_scores_short.max() - bm25_scores_short.min():.2f}\n")

print("⚠️ Issue: Different document lengths create incomparable score ranges\n")

print("=== FAILURE 3: QUERY TYPE MISMATCH ===\n")

# Query that's purely semantic
semantic_query = "device for listening to music wirelessly"
keyword_query = "WH-1000XM5"

print(f"Semantic query: '{semantic_query}'")
results_sem = search_products(semantic_query, alpha=0.5, top_k=2)
for r in results_sem:
    print(f"  {r['product']['name']} (BM25: {r['bm25_score']:.3f}, Sem: {r['semantic_score']:.3f})")

print(f"\nKeyword query: '{keyword_query}'")
results_key = search_products(keyword_query, alpha=0.5, top_k=2)
for r in results_key:
    print(f"  {r['product']['name']} (BM25: {r['bm25_score']:.3f}, Sem: {r['semantic_score']:.3f})")

print("\n⚠️ Issue: Fixed alpha doesn't adapt to query type\n")

print("=== SOLUTIONS ===")
print("""
1. Learned Alpha: Train model to predict optimal alpha per query
2. Query Classification: Route queries to optimal retrieval method
3. Score Calibration: Use learned combinations or RRF
4. Adaptive Fusion: Dynamic weighting based on query signals
5. Ensemble Methods: Combine multiple fusion strategies
""")

## 📊 Benchmark Comparison

| Retrieval Method | NDCG@10 | Recall@10 | Latency | Best For |
|------------------|---------|-----------|---------|----------|
| **BM25 Only** | 0.45 | 0.52 | 10ms | Exact matches |
| **Semantic Only** | 0.58 | 0.61 | 150ms | Conceptual queries |
| **Linear Hybrid** | 0.67 | 0.71 | 160ms | General purpose |
| **RRF Hybrid** | 0.69 | 0.74 | 160ms | Robust ranking |
| **Learned Fusion** | 0.73 | 0.78 | 200ms | Production systems |

### Alpha Tuning Guidelines:
| Query Type | Recommended Alpha | Rationale |
|------------|-------------------|-----------|
| Product IDs/Codes | 0.8-1.0 | Exact match critical |
| Brand + Product | 0.6-0.7 | Balance brand match |
| Feature Description | 0.2-0.4 | Semantic understanding |
| Natural Language | 0.3-0.5 | Concept matching |
| Technical Specs | 0.7-0.9 | Precise matching |

### Key Insights:
- Hybrid consistently outperforms single-method approaches
- RRF is more robust than linear combination
- Query-aware alpha tuning provides 5-10% improvement
- BM25 adds minimal latency overhead

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              HYBRID RETRIEVAL EXPERIMENT LAB                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Hybrid Retrieval Playground\n")
print("Using the product catalog from the real-world example.\n")

while True:
    query = input("\nEnter search query (or 'quit' to exit): ")
    if query.lower() == 'quit':
        break
    
    print("\nChoose alpha (weight for BM25):")
    print("  0.0 = Semantic only")
    print("  0.5 = Balanced (default)")
    print("  1.0 = BM25 only")
    alpha_input = input("Alpha (0.0-1.0, default 0.5): ")
    alpha = float(alpha_input) if alpha_input else 0.5
    
    top_k = int(input("Number of results (default 5): ") or "5")
    
    results = search_products(query, alpha=alpha, top_k=top_k)
    
    print(f"\n{'='*70}")
    print(f"Results for: '{query}' (alpha={alpha})")
    print(f"{'='*70}\n")
    
    print(f"{'Rank':<6}{'Product':<25}{'Combined':<10}{'BM25':<8}{'Semantic':<10}")
    print("-" * 70)
    
    for i, r in enumerate(results, 1):
        product = r['product']
        name = product['name'][:23] + ".." if len(product['name']) > 25 else product['name']
        print(f"{i:<6}{name:<25}{r['combined_score']:<10.3f}{r['bm25_score']:<8.3f}{r['semantic_score']:<10.3f}")
    
    # Show detailed view option
    detail = input("\nShow details? (enter rank number or 'n'): ")
    if detail.isdigit():
        rank = int(detail)
        if 1 <= rank <= len(results):
            r = results[rank-1]
            p = r['product']
            print(f"\n{'='*70}")
            print(f"Product Details: {p['name']}")
            print(f"{'='*70}")
            print(f"ID: {p['id']}")
            print(f"Description: {p['description']}")
            print(f"\nScores:")
            print(f"  Combined:  {r['combined_score']:.4f}")
            print(f"  BM25:      {r['bm25_score']:.4f}")
            print(f"  Semantic:  {r['semantic_score']:.4f}")

## 💡 Tips & Tricks

### Advanced Fusion Techniques:

**1. Reciprocal Rank Fusion (RRF):**
```python
def rrf_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)
```

**2. Query-Adaptive Alpha:**
```python
def adaptive_alpha(query):
    if has_product_code(query): return 0.9
    if is_natural_language(query): return 0.3
    return 0.5  # default
```

**3. Learned Fusion (LightGBM/XGBoost):**
- Train on query-document features
- Combine BM25 score, semantic score, query length, etc.
- Often achieves best performance

### Production Considerations:
- **Caching**: Cache embeddings and BM25 scores
- **Batching**: Batch embedding requests for efficiency
- **Filtering**: Pre-filter by category before hybrid search
- **Monitoring**: Track which method contributes more per query type

### Common Mistakes:
- ❌ Not normalizing scores before combining
- ❌ Using fixed alpha for all query types
- ❌ Ignoring latency impact of dual retrieval
- ❌ Not handling cases where one method returns no results

## 📚 References

### Research:
- [Dense Retrieval Meets Sparse Retrieval (Lin et al., 2021)](https://arxiv.org/abs/2106.11267)
- [Reciprocal Rank Fusion (Cormack et al., 2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [COIL: Contextualized Exact Match (Gao & Callan, 2021)](https://arxiv.org/abs/2104.07186)

### Documentation:
- [BM25 Algorithm](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)
- [Pinecone Hybrid Search](https://docs.pinecone.io/docs/hybrid-search)
- [Weaviate Hybrid Search](https://weaviate.io/developers/weaviate/search/hybrid)

### Related Techniques:
- Semantic Search (Technique 56)
- Re-Ranking (Technique 58)
- Query Expansion (Technique 59)
- Basic RAG (Technique 53)